# SWASTHYAGUIDE 2.0 - 03. Model Validation & Explainability

**Stage 3.** Repeated cohort-aware cross-validation, a bounded hyper-parameter
search, threshold and calibration analysis, a `ca` feature-availability
ablation, error analysis, feature attribution, and one final look at the locked
holdout.

**Scope disclaimer.** Research prototype for health **risk assessment**. The
model outputs are *research risk scores*, not clinically validated
probabilities. Not a medical device, not a diagnostic system, no clinical
deployment. Every result below requires external validation before any use.

**Data rule.** `data/heart_disease_uci.csv` is read-only. Nothing in this
notebook writes to it; the SHA-256 is re-verified at the end.

**How to reproduce.** Run `python -m src.validate_stage3` once. This notebook
demonstrates the method live where it is cheap, and loads the persisted result
tables for the parts that take minutes to recompute.

## 1. Imports

The logic lives in `src/validation.py` and `src/explainability.py`; this notebook
imports them so what is shown is what ships.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src" / "validation.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("src/validation.py not found")
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src import preprocessing as prep
from src import validation as val
from src import explainability as exp
from src import verify_stage3
from sklearn.base import clone

RANDOM_SEED = prep.RANDOM_SEED
TEST_SIZE = 0.20
THRESHOLD = 0.50          # reporting threshold, not a clinical cut-off
MODEL_ORDER = list(prep.BASELINE_MODELS)
PRETTY = {
    "logistic_regression": "Logistic Regression",
    "random_forest": "Random Forest",
    "xgboost": "XGBoost",
}

print("seed:", RANDOM_SEED, "| cv:", val.CV_N_SPLITS, "x", val.CV_N_REPEATS)
print("threshold grid:", val.THRESHOLD_GRID)

## 2. What Stage 2 could not tell us

Stage 2 reported one 80/20 split: 184 test rows, one draw, one number per model.
Three problems follow from that, and each is what Stage 3 is built to remove.

1. **A single split has no spread.** The difference between the three Stage 2
   ROC-AUC values is 0.019. Without fold-to-fold variation there is no way to
   know whether that gap means anything.
2. **One threshold, no sensitivity/specificity trade-off.** Reporting only at
   0.50 hides the whole recall-versus-specificity curve.
3. **A single number per cohort mixes two different failure modes** - model
   error, and a cohort with 93.5% prevalence where almost every row is positive.

Everything below keeps the Stage 2 holdout locked. Repeated cross-validation
runs on the 736 training rows only, and the 184 holdout rows are scored once, at
the very end, after every decision is frozen.

## 3. The locked split and the fold construction

The Stage 2 split is re-derived, not redefined: `test_size=0.20`,
`random_state=42`, stratified on the **joint cohort x label** key. Folds are then
built from the training portion alone, stratified on that same key, so every
cohort appears in every fold with a comparable class balance.

Note the assertion: a `cohort x label` stratum with fewer rows than
`n_splits` would silently break stratification, so the fold builder raises
instead.

In [ ]:
df = prep.build_analysis_frame()
X, y, cohorts = prep.split_xy(df)

train_idx, test_idx = val.build_locked_split(df, TEST_SIZE, RANDOM_SEED)
X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]
cohorts_train, cohorts_test = cohorts.iloc[train_idx], cohorts.iloc[test_idx]

strata_train = val.composite_strata(cohorts_train, y_train)
cv_splits = val.make_cv_splits(strata_train, val.CV_N_SPLITS, val.CV_N_REPEATS, RANDOM_SEED)

# Fold positions are 0..735, i.e. positions WITHIN the training frame, so they
# cannot be compared to test_idx directly. Map each fold position back to its
# original row in the 920-row frame first - otherwise the overlap test compares
# two different coordinate systems and trivially "passes".
train_idx = np.asarray(train_idx)
test_idx = np.asarray(test_idx)
rows_used_as_validation = set(train_idx[np.concatenate([va for _, va in cv_splits])].tolist())
rows_used_for_training = set(train_idx[np.concatenate([tr for tr, _ in cv_splits])].tolist())
holdout_rows = set(test_idx.tolist())

print(f"train {len(X_train)} | test {len(X_test)} | folds {len(cv_splits)}")
print("holdout rows validated in some fold:", len(holdout_rows & rows_used_as_validation))
print("holdout rows used to fit some fold:", len(holdout_rows & rows_used_for_training))
print("every training row is validated somewhere:",
      rows_used_as_validation == set(train_idx.tolist()))

assert not (holdout_rows & rows_used_as_validation), "a holdout row reached a fold's validation set"
assert not (holdout_rows & rows_used_for_training), "a holdout row was used to fit a fold"
assert rows_used_as_validation == set(train_idx.tolist()), "some training row was never validated"

coverage = pd.Series(np.concatenate([va for _, va in cv_splits])).value_counts()
print(f"rows validated: {len(coverage)}/{len(X_train)} | "
      f"min times {coverage.min()} (one per repeat), max {coverage.max()}")

pd.DataFrame({
    "cohort x label stratum": strata_train.value_counts(),
}).sort_index()

Each training row is validated exactly once per repeat, so with 5 repeats it
receives 5 out-of-fold scores. `run_cross_validation` averages them. Averaging
is not cosmetic: before this was fixed, a row's pooled prediction was whichever
repeat happened to be fitted last, which silently discarded four fifths of the
work.

In [ ]:
probe = val.build_pipeline_for("logistic_regression", *val.feature_set(),
                                 random_state=RANDOM_SEED)
per_fold_probe, oof_one_repeat, counts = val.run_cross_validation(
    probe, X_train, y_train, cv_splits[:val.CV_N_SPLITS], THRESHOLD
)
per_fold_full, oof_all_repeats, counts_full = val.run_cross_validation(
    probe, X_train, y_train, cv_splits, THRESHOLD
)

print("single repeat -> counts per row:", sorted(set(counts.tolist())))
print("all repeats   -> counts per row:", sorted(set(counts_full.tolist())))
print("mean out-of-fold score (1 repeat):", round(float(oof_one_repeat.mean()), 4))
print("mean out-of-fold score (5 repeats):", round(float(oof_all_repeats.mean()), 4))
print("averaging changes a row's score by up to:",
      round(float(np.abs(oof_all_repeats - oof_one_repeat).max()), 4))

per_fold_full[["fold", "roc_auc", "pr_auc", "recall", "precision", "f1"]].round(4)

## 4. Leakage safety, demonstrated rather than asserted

Everything that learns from data - medians, scaler, one-hot categories - lives
inside the `Pipeline`. Each fold gets a fresh `clone`, so those statistics are
re-fitted on that fold's training rows only. Below, the imputer medians from a
single fold's fit are compared against medians computed three ways: fold-train,
full-train, and the holdout. If fold-train differs from full-train, leakage is
present and the check fails.

In [ ]:
fold_train, fold_val = cv_splits[0]
fold_pipeline = clone(probe)
fold_pipeline.fit(X_train.iloc[fold_train], y_train.iloc[fold_train].to_numpy())

num_pipe = fold_pipeline.named_steps["preprocess"].named_transformers_["numeric"]
imputer = num_pipe.named_steps["impute_median"]
masker = num_pipe.named_steps["mask_invalid_values"]

# The numeric branch receives columns in pipeline order. statistics_ follows
# that same order, so the index must be num_cols and NOT an alphabetical sort -
# comparing the two series would silently misalign them.
num_cols = list(prep.NUMERIC_FEATURES) + list(prep.BOOLEAN_SOURCE_COLS)
assert len(imputer.statistics_) == len(num_cols), "numeric branch width mismatch"
fitted_medians = pd.Series(imputer.statistics_, index=num_cols)


def medians_for(frame):
    masked = masker.transform(frame[num_cols].to_numpy(dtype=float))
    return pd.Series(np.nanmedian(masked, axis=0), index=num_cols)


observed = medians_for(X_train.iloc[fold_train])
full_train = medians_for(X_train)
holdout = medians_for(X_test)

comparison = pd.DataFrame({
    "fold_fit_median": fitted_medians,
    "recomputed_fold_train": observed,
    "full_train": full_train,
    "holdout": holdout,
})
comparison.index.name = "column"
comparison["matches_fold_train"] = np.isclose(comparison.fold_fit_median,
                                              comparison.recomputed_fold_train)
comparison["differs_from_full_train"] = ~np.isclose(comparison.fold_fit_median,
                                                    comparison.full_train)

assert comparison["matches_fold_train"].all(), "imputer medians are not fold-train medians"
assert comparison["differs_from_full_train"].any(), "check cannot detect leakage if values coincide"
print("imputer medians equal fold-train medians:", bool(comparison.matches_fold_train.all()))
print("and at least one differs from the full-train median, so this check can fail")

comparison

In [ ]:
preprocessor = fold_pipeline.named_steps["preprocess"]
transformed = preprocessor.transform(X_train.iloc[fold_val])
print("validation rows in :", len(fold_val))
print("validation rows out:", transformed.shape[0])
print("NaN surviving preprocessing:", int(np.isnan(transformed).sum()))
print("transformed columns:", transformed.shape[1])
print("holdout rows pass through the same code path:", preprocessor.transform(X_test).shape)

## 5. Repeated cross-validation - un-tuned baselines

Same 25 folds for all three models, mean +/- standard deviation across folds.
These are the Stage 2 defaults, so this section isolates the effect of
repeated CV alone. The per-fold rows are also persisted to
`models/cv_metrics_per_fold.csv`, which is what makes the paired comparison at
the end of this section possible.

In [ ]:
cv_overall = pd.read_csv(prep.MODELS_DIR / "cv_metrics_overall.csv")
cv_cohort = pd.read_csv(prep.MODELS_DIR / "cv_metrics_by_cohort.csv")

summary_cols = ["model", "n_folds"] + [f"{m}_{s}" for m in val.CV_METRIC_NAMES
                                      for s in ("mean", "sd")]
cv_overall[summary_cols].round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
x = np.arange(len(cv_overall))
width = 0.35
for ax, (mean_col, sd_col, title, colours) in zip(axes, [
    ("roc_auc", "#4c72b0", "ROC-AUC, mean +/- SD over 25 folds", "#4c72b0"),
    ("pr_auc", "#55a868", "PR-AUC, mean +/- SD over 25 folds", "#55a868"),
]):
    ax.bar(x, cv_overall[f"{mean_col}_mean"], width, yerr=cv_overall[f"{mean_col}_sd"],
           capsize=4, color=colours, label=mean_col.upper())
    ax.set_xticks(x)
    ax.set_xticklabels([PRETTY[m] for m in cv_overall["model"]], fontsize=8)
    ax.set_ylim(0.82, 0.98)
    ax.set_title(title, fontsize=10)
    ax.grid(axis="y", alpha=0.25)
    ax.legend(fontsize=8)

# Threshold-dependent metrics need no error bars here: the SD is dominated by
# fold difficulty rather than by anything the plot would convey at this size.
cv_overall.set_index("model")[["recall_mean", "precision_mean", "f1_mean"]].rename(
    columns={"recall_mean": "recall", "precision_mean": "precision", "f1_mean": "F1"}
).plot.bar(ax=axes[1], color=["#4c72b0", "#dd8452", "#55a868"])
axes[1].set_title(f"At threshold {THRESHOLD}", fontsize=10)
axes[1].set_ylim(0.75, 0.92)
axes[1].tick_params(axis="x", rotation=0, labelsize=8)
axes[1].grid(axis="y", alpha=0.25)
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

### Comparing models: pair the folds

Comparing one model's SD against another model's mean is **not** a valid test
here. All three models run on the *same* 25 folds, so fold difficulty - which
row lands in which validation set - is shared and largely cancels. The paired
per-fold difference is the correct uncertainty estimate, and it is much tighter
than the summary SDs suggest.

In [ ]:
# Rebuilt from the persisted per-fold rows, so these numbers come from the
# artefacts rather than a fresh in-memory run.
per_fold = pd.read_csv(prep.MODELS_DIR / "cv_metrics_per_fold.csv")
wide = per_fold.pivot(index="fold", columns="model", values="roc_auc")


def paired_gaps(frame, label):
    rows = []
    for i, left in enumerate(MODEL_ORDER):
        for right in MODEL_ORDER[i + 1:]:
            diff = (frame[left] - frame[right]).dropna()
            mean_diff, sd_diff = float(diff.mean()), float(diff.std(ddof=1))
            wins = int((diff > 0).sum())
            rows.append({
                "comparison": f"{PRETTY[left]} - {PRETTY[right]}",
                "mean gap": round(mean_diff, 4),
                "paired SD": round(sd_diff, 4),
                "folds first better": f"{wins}/{len(diff)}",
                "gap > paired SD": bool(abs(mean_diff) >= sd_diff
                                        and max(wins, len(diff) - wins) / len(diff) >= 0.8),
            })
    return pd.DataFrame(rows)


paired_baseline = paired_gaps(wide, "un-tuned")
paired_baseline

**Reading this honestly.** Against the paired test the un-tuned defaults *are*
separated: Logistic Regression beats XGBoost on 24 of 25 folds, and the 0.031
gap is more than twice its paired SD. That is a real ordering on this dataset.

It is still not a clinical ranking. Logistic Regression leads partly because it
is the simplest model on a dataset this small and this noisy, ROC-AUC on one
dataset says nothing about a different population, and - as section 7 shows -
most of that ordering disappears once the models are tuned. Doing the pairing
first is what makes that visible instead of hiding it behind a large SD.

A p-value is deliberately not reported: with 25 folds the binomial reference is
coarse, and a significance test here invites exactly the "this model is better"
reading this project cannot support.

## 6. Bounded hyper-parameter search

`RandomizedSearchCV` on the **same** folds, scored on ROC-AUC but reporting five
metrics per candidate so the selection can be sanity-checked against more than
one number. Two budget decisions are deliberate and are recorded rather than
hidden:

* The search uses the **first repeat only** (5 folds, not 25), to keep the
  runtime bounded.
* Where the grid is larger than the budget, sampling is **with replacement**.
  Some combinations are drawn more than once and others not at all, so for
  Random Forest and XGBoost the search is explicitly *partial* coverage. The
  output says so per model rather than implying an exhaustive grid.

In [ ]:
card = __import__("json").loads((prep.MODELS_DIR / "stage3_report.json").read_text(encoding="utf-8"))
search_rows = []
for model_name in MODEL_ORDER:
    s = card["search"][model_name]
    search_rows.append({
        "model": model_name,
        "combinations": s["space"]["total_combinations"],
        "sampled": s["space"]["n_iter_sampled"],
        "unique_scored": s["n_unique_combinations_scored"],
        "coverage": "exhaustive" if s["space_exhaustively_covered"] else "partial",
        "search_folds": s["space"]["cv_folds_used"],
        "best_cv_roc_auc": s["best_cv_roc_auc"],
        "best_params": ", ".join(f"{k}={v}" for k, v in s["best_params"].items()),
    })
search_table = pd.DataFrame(search_rows)
search_table.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.6))
labels = [PREPY_NAME if (PREPY_NAME := PRETTY[m]) else m for m in search_table["model"]]
ax.barh(labels, search_table["combinations"], color="#c44e52", alpha=0.35,
        label="combinations in the grid")
ax.barh(labels, search_table["unique_scored"], color="#4c72b0",
        label="combinations actually scored")
ax.set_xlabel("count")
ax.set_title("Search coverage: budget is smaller than the grid for two models")
ax.legend()
plt.tight_layout()
plt.show()

**Reading this honestly.** The selected configurations are ordinary
settings - moderate regularisation, shallow trees, low learning rate. Nothing
here is a tuned discovery. A better search CV score is not evidence of clinical
superiority; it is one number from 5 folds.

## 7. Repeated cross-validation of the selected configurations

Same 25 folds, same procedure, with each model's selected hyper-parameters. The
deltas against the un-tuned baseline are small everywhere, which is itself the
result worth stating: **tuning did not meaningfully move any model.**

This is also the prediction set used for every downstream development analysis
below - thresholds, calibration, error analysis and case selection all describe
these selected models, not the un-tuned defaults.

In [ ]:
tuned_overall = pd.read_csv(prep.MODELS_DIR / "cv_metrics_tuned_overall.csv")
tuned_cohort = pd.read_csv(prep.MODELS_DIR / "cv_metrics_tuned_by_cohort.csv")

delta = tuned_overall[["model"] + [f"{m}_mean" for m in val.CV_METRIC_NAMES]].merge(
    cv_overall[["model"] + [f"{m}_mean" for m in val.CV_METRIC_NAMES]],
    on="model", suffixes=("_tuned", "_baseline"),
)
delta_table = pd.DataFrame({
    "model": delta["model"],
    "roc_auc baseline": delta["roc_auc_mean_baseline"].round(4),
    "roc_auc selected": delta["roc_auc_mean_tuned"].round(4),
    "roc_auc delta": (delta["roc_auc_mean_tuned"] - delta["roc_auc_mean_baseline"]).round(4),
    "pr_auc delta": (delta["pr_auc_mean_tuned"] - delta["pr_auc_mean_baseline"]).round(4),
    "recall delta": (delta["recall_mean_tuned"] - delta["recall_mean_baseline"]).round(4),
    "precision delta": (delta["precision_mean_tuned"] - delta["precision_mean_baseline"]).round(4),
    "f1 delta": (delta["f1_mean_tuned"] - delta["f1_mean_baseline"]).round(4),
})
delta_table

**Tuning collapses the separation between the models.** The paired test from
section 5 is worth repeating on the selected configurations, because this is
where the un-tuned ordering stops holding.

In [ ]:
per_fold_tuned = pd.read_csv(prep.MODELS_DIR / "cv_metrics_tuned_per_fold.csv")
paired_tuned = paired_gaps(
    per_fold_tuned.pivot(index="fold", columns="model", values="roc_auc"), "selected"
)
paired_tuned

Two of the three separations disappear: Logistic Regression vs Random Forest
now wins only 20 of 25 folds and its 0.006 gap is *smaller* than its own paired
SD, and Random Forest vs XGBoost wins 15 of 25 folds - effectively a coin flip.
Only Logistic Regression vs XGBoost still clears the bar, at 22 of 25 folds and a
0.008 gap.

The practical reading is that on this dataset the three models are close to
interchangeable on discrimination, and the honest choice among them rests on
calibration and interpretability (sections 9 and 11) rather than on ROC-AUC.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
melted = delta_table.melt(id_vars="model", value_name="delta", var_name="metric")
sns.barplot(data=melted, x="metric", y="delta", hue="model", ax=ax)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Selected vs un-tuned configuration: change in mean CV metric")
ax.set_ylabel("change (selected - baseline)")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=20, labelsize=8)
plt.tight_layout()
plt.show()

The largest movement is XGBoost ROC-AUC +0.024. Note this is a *pre-selection*
comparison against the baseline mean, not a test: the deltas were computed on the
same data that chose the hyper-parameters, so they are optimistically biased and
cannot be read as evidence that tuning helped.

## 8. Cross-validation by cohort

Pooled out-of-fold metrics per source cohort, with an explicit `reliable` flag:
a slice needs at least 50 rows, at least 10 positives **and** at least 10
negatives, or the uncertainty around the figure is too wide to act on.

The flag is computed from the data in code and the verification suite re-derives
it independently - it is not a hand-written list of cohort names.

Note the two flags differ by population size. In the 736-row cross-validation
pool VA Long Beach has 160 rows and 41 negatives, so it clears the bar there.
In the 184-row holdout it drops to 40 rows, where it does not.

In [ ]:
cohort_view = tuned_cohort[["model", "cohort", "n", "n_positive", "prevalence",
                              "roc_auc", "pr_auc", "recall", "precision", "f1", "reliable"]]
cohort_view.round(4)

In [ ]:
# The flag re-derived from the stated criteria, not read from the CSV column.
criteria = val.MIN_COHORT_ROWS, val.MIN_COHORT_CLASS_COUNT
derived = (
    (cohort_view["n"] >= criteria[0])
    & (cohort_view["n_positive"] >= criteria[1])
    & ((cohort_view["n"] - cohort_view["n_positive"]) >= criteria[1])
)
print(f"criteria: n >= {criteria[0]}, positives >= {criteria[1]}, negatives >= {criteria[1]}")
print("flag matches the criteria on every row:", bool((derived == cohort_view["reliable"]).all()))
print("flagged cohorts:", sorted(cohort_view.loc[~derived, "cohort"].unique().tolist()))
print("holdout sizes for the same cohorts:")
print(pd.Series({"Switzerland": int((cohorts_test == "Switzerland").sum()),
                 "VA Long Beach": int((cohorts_test == "VA Long Beach").sum())}).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))
pivot = tuned_cohort.pivot(index="cohort", columns="model", values="roc_auc")
order = tuned_cohort.groupby("cohort")["prevalence"].first().sort_values().index
pivot.loc[order].plot.bar(ax=ax, color=["#4c72b0", "#55a868", "#c44e52"])
ax.axhline(0.5, linestyle="--", color="grey", linewidth=1, label="chance")
ax.set_ylim(0.4, 1.0)
ax.set_ylabel("out-of-fold ROC-AUC")
ax.set_title("Discrimination by cohort (pooled out-of-fold, selected models)")
ax.tick_params(axis="x", rotation=15)
ax.legend(fontsize=8, title="Switzerland slice is flagged unreliable", title_fontsize=7)
plt.tight_layout()
plt.show()

**Reading this honestly.** Cleveland and Hungary - the two cohorts with
enough rows on both sides - sit around ROC-AUC 0.89-0.92 for all three models.
Switzerland and VA Long Beach sit near 0.69-0.76, and both carry high
prevalence (93.9% and 74.4%). When almost every row is positive, ROC-AUC has
little room to move and the metric stops being informative. The pooled figure
near 0.90 is an average over populations that are not clinically comparable, and
it should not be quoted without this paragraph attached.

No conclusion is drawn from Switzerland (98 rows, 6 negatives, flagged
unreliable) or from the 40-row VA Long Beach holdout slice.

## 9. Threshold analysis

The full grid `0.30, 0.40, 0.50, 0.60, 0.70` is reported for every model. **No
threshold is selected, tuned or recommended here.** The cut-off is a research
parameter whose value depends on what a false negative and a false positive each
cost in the intended setting - and this dataset cannot answer that question.
0.50 is used only because it is the conventional reporting default.

In [ ]:
thresholds = pd.read_csv(prep.MODELS_DIR / "threshold_analysis.csv")
thresholds[["model", "threshold", "sensitivity_recall", "specificity", "precision",
            "f1", "tn", "fp", "fn", "tp"]].round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
for model_name in MODEL_ORDER:
    part = thresholds[thresholds["model"] == model_name]
    label = PRETTY[model_name]
    axes[0].plot(part["threshold"], part["sensitivity_recall"], marker="o", label=label)
    axes[0].plot(part["threshold"], part["specificity"], marker="s", linestyle="--",
                 alpha=0.5, color=axes[0].lines[-1].get_color())
    axes[1].plot(part["threshold"], part["f1"], marker="o", label=label)

axes[0].set_xlabel("threshold"); axes[0].set_ylabel("rate")
axes[0].set_title("Solid: sensitivity (recall)   Dashed: specificity")
axes[1].set_xlabel("threshold"); axes[1].set_ylabel("F1")
axes[1].set_title("F1 across the same grid")
for ax in axes:
    ax.set_xticks(val.THRESHOLD_GRID)
    ax.grid(alpha=0.25)
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

print("counts at the reporting threshold", THRESHOLD)
print(thresholds[thresholds["threshold"] == THRESHOLD][
    ["model", "tn", "fp", "fn", "tp", "sensitivity_recall", "specificity"]
].to_string(index=False))

**Reading this honestly.** Lowering the cut-off to 0.30 buys about 9 extra
true positives per 736 rows and costs about 43 extra false positives. That trade
is a policy decision for a future clinical setting, not something this dataset
settles. F1 peaks around 0.50 for all three models, which tells us the default
is near the balance point of *this* metric - it does not make 0.50 correct.

## 10. Calibration

A risk score is only interpretable if it means what it says. Four measurements:

* **Brier score** - mean squared error of the score, against a constant
  prevalence baseline. All three roughly halve it.
* **Expected calibration error** - population-weighted mean gap between score
  and observed frequency within bins.
* **Slope / intercept** from a logistic recalibration fit. Ideal is 1 and 0.
* **The reliability diagram** below - observed frequency against mean predicted
  score, with the diagonal marking perfect agreement.

These are **research risk scores**. Agreement inside this dataset is not
evidence of clinically validated probability, and none of these numbers should
be quoted as a real-world event rate.

In [ ]:
calibration = pd.read_csv(prep.MODELS_DIR / "calibration_metrics.csv")
calibration[["model", "n", "prevalence", "mean_predicted_probability", "brier_score",
             "brier_baseline_constant_prevalence", "expected_calibration_error",
             "calibration_slope", "calibration_intercept"]].round(4)

In [ ]:
bins = pd.read_csv(prep.MODELS_DIR / "calibration_bins.csv")

fig, ax = plt.subplots(figsize=(6.6, 6.2))
for model_name in MODEL_ORDER:
    part = bins[(bins["model"] == model_name) & (bins["n"] > 0)]
    ax.plot(part["mean_predicted"], part["observed_frequency"], marker="o",
            linewidth=1.6, label=PRETTY[model_name])
ax.plot([0, 1], [0, 1], linestyle="--", color="grey", linewidth=1.2,
        label="perfect agreement")
ax.set_xlabel("mean predicted risk score")
ax.set_ylabel("observed outcome frequency")
ax.set_title("Calibration of out-of-fold risk scores (selected models)")
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.grid(alpha=0.25); ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

saved = prep.MODELS_DIR / "calibration_curve.png"
print("the driver writes this figure to models/calibration_curve.png")

**Reading this honestly.** Logistic Regression and XGBoost land close to the
ideal slope of 1.0 with intercepts near zero. Random Forest has slope 1.33,
i.e. its scores are under-dispersed - the extremes are not extreme enough -
and its expected calibration error is twice the others. Every model also
over-predicts risk in the middle of the range and under-predicts at both ends,
which is visible as points sitting alternately above and below the diagonal.

If calibrated probabilities were a requirement, the Random Forest score would
need recalibration before use. It is not recalibrated here, because doing so
would be a modelling decision made after seeing these results.

## 11. Feature-availability ablation: what `ca` is worth

`ca` is an angiographic count of narrowed vessels. It is the strongest single
predictor in the dataset, and it is also 66.4% missing and almost entirely
recorded for Cleveland only. That combination is a deployment question, not a
modelling one: an ordinary pre-screening interaction cannot run angiography on
everyone.

So the same three model families, the same folds and the same procedure were
re-run with `ca` removed from all three feature lists. **This measures the cost
of the missing feature. It is not a claim that either variant is clinically
preferable.**

In [ ]:
ablation = pd.read_csv(prep.MODELS_DIR / "feature_ablation_ca.csv")
ablation_wide = ablation.pivot(index="model", columns="feature_variant")
ablation_wide[[
    ("roc_auc_mean", "full_13_features"), ("roc_auc_mean", "no_ca_12_features"),
    ("roc_auc_sd", "full_13_features"), ("roc_auc_sd", "no_ca_12_features"),
]].round(4)

In [ ]:
full_variant = f"full_{len(prep.FEATURE_COLUMNS)}_features"
no_ca_variant = f"no_{val.ABLATION_REMOVED_FEATURE}_{len(prep.FEATURE_COLUMNS) - 1}_features"

cost = []
for model_name in MODEL_ORDER:
    part = ablation[ablation["model"] == model_name].set_index("feature_variant")
    cost.append({
        "model": model_name,
        "roc_auc with ca": part.loc[full_variant, "roc_auc_mean"],
        "roc_auc without ca": part.loc[no_ca_variant, "roc_auc_mean"],
        "roc_auc cost": part.loc[no_ca_variant, "roc_auc_mean"] - part.loc[full_variant, "roc_auc_mean"],
        "pr_auc cost": part.loc[no_ca_variant, "pr_auc_mean"] - part.loc[full_variant, "pr_auc_mean"],
        "f1 cost": part.loc[no_ca_variant, "f1_mean"] - part.loc[full_variant, "f1_mean"],
        "paired sd (roc_auc)": part.loc[no_ca_variant, "roc_auc_sd"],
    })
cost_table = pd.DataFrame(cost)
cost_table.iloc[:, 1:] = cost_table.iloc[:, 1:].round(4)
cost_table

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
melted = cost_table.melt(id_vars="model", value_name="cost",
                         var_name="metric", value_vars=["roc_auc cost", "pr_auc cost", "f1 cost"])
sns.barplot(data=melted, x="metric", y="cost", hue="model", ax=ax)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title(f"Cost of removing `ca` (mean over 25 folds)")
ax.set_xlabel("")
ax.set_ylabel("change (without ca - with ca)")
ax.tick_params(axis="x", rotation=15, labelsize=8)
plt.tight_layout()
plt.show()

**Reading this honestly.** Removing `ca` costs every model between 0.005 and
0.008 ROC-AUC. That is a real but modest price - smaller than the fold-to-fold
spread of a single model. The ablation also matters because the missingness
itself is informative: rows without `ca` have higher disease prevalence (59.6%
vs 46.8%), which is why the `missingindicator_ca` column is among the strongest
features for Logistic Regression.

Note this is a *paired* comparison - identical folds, identical models, one
feature list changed - so the comparison is tighter than the between-model
differences elsewhere in this notebook. It still describes the dataset, not a
future deployment.

## 11b. Error analysis

Out-of-fold errors on the 736 training rows, sliced by cohort, missingness and
the main features. **These are associations within the data. They are not causal
claims and carry no clinical interpretation.**

In [ ]:
errors = card["error_analysis"]
rows = []
for model_name in MODEL_ORDER:
    counts = errors["models"][model_name]
    rows.append({
        "model": model_name,
        "TN": counts["tn"], "FP": counts["fp"], "FN": counts["fn"], "TP": counts["tp"],
        "error_rate": round(counts["error_rate"], 4),
        "FN share of errors": round(counts["fn_share_of_errors"], 4),
        "mean score if FN": round(counts["mean_prob_by_outcome"]["FN"], 3),
        "mean score if FP": round(counts["mean_prob_by_outcome"]["FP"], 3),
        "mean score if TN": round(counts["mean_prob_by_outcome"]["TN"], 3),
        "mean score if TP": round(counts["mean_prob_by_outcome"]["TP"], 3),
    })
pd.DataFrame(rows)

In [ ]:
for group_name in ["dataset", "missing_any", "sex"]:
    print(f"--- error rate by {group_name} ---")
    table = pd.DataFrame(errors["groups"]["logistic_regression"][group_name])
    print(table.round(4).to_string(index=False))
    print()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, group_name in zip(axes, ["dataset", "missing_any"]):
    table = pd.DataFrame(errors["groups"]["logistic_regression"][group_name])
    sns.barplot(data=table, x="level", y="error_rate", ax=ax, color="#4c72b0")
    ax.set_ylim(0, max(0.35, float(table["error_rate"].max()) * 1.25))
    ax.set_xlabel("")
    ax.set_ylabel("out-of-fold error rate")
    ax.set_title(f"Error rate by {group_name} (Logistic Regression)")
    ax.tick_params(axis="x", rotation=15, labelsize=8)
plt.tight_layout()
plt.show()

**Reading this honestly.** Three patterns are visible and all three are
associations, not causes:

* **FN and FP are roughly balanced** for every model, so the errors are not
  dominated by one failure mode.
* **VA Long Beach carries the largest error rate** (~0.24) for all three models,
  and its false positives are the main contributor. Its 74.4% prevalence leaves
  little room to be wrong.
* **Rows with missing values are not clearly harder for the tuned models.** The
  early gap seen with the un-tuned baselines largely disappears once missingness
  indicators and train-only imputation are in place. Females have a lower error
  rate than males (0.12 vs 0.19), but the female slice is also 79% non-disease,
  so part of that difference is base rate rather than model skill.

## 12. Explainability

Three views of the same thing, in increasing order of model-specificity:

1. **Logistic Regression coefficients** - directly interpretable, one number per
   transformed feature, sign is the direction of the effect on the risk score.
2. **Impurity-based importance** for the tree models - biased towards
   high-cardinality features, so it is read as a rough ranking only.
3. **SHAP values** - additive per-feature contributions relative to a baseline,
   available for all three models and comparable *within* a model.

**Three output-scale and baseline traps**, each of which silently produces
plausible-looking but wrong numbers if ignored. All three are handled in
`src/explainability.py` and verified numerically.

1. **The units differ by model, and the obvious guess is wrong.** shap
   decomposes Logistic Regression in **log-odds** and Random Forest in
   **probability** units - but XGBoost is also in **log-odds**, not probability.
   With the default `tree_path_dependent` perturbation `TreeExplainer` only
   supports `model_output="raw"`, so an XGBoost *classifier* is decomposed on
   its raw margin while `predict_proba` returns a probability. A hard-coded
   "tree models are in probability units" label would misreport XGBoost by
   comparing log-odds against probabilities.

   Rather than assume, the code tests which model output
   `base + sum(contributions)` reproduces, and records which one matched:

   | Model | Verified against | Scale | Max gap |
   |-------|------------------|-------|---------|
   | Logistic Regression | `decision_function` (log-odds) | log-odds | 1.8e-15 |
   | Random Forest | `predict_proba` | probability | 1.4e-15 |
   | XGBoost | raw margin (log-odds) | log-odds | 4.7e-06 |

   Magnitudes are therefore **not comparable across model families** - only
   signs and within-model rankings are.

2. **The baseline is the per-row expected output, not the mean SHAP value.**
   Those differ, and using the mean makes every waterfall's top segment a number
   that does not equal the model's own prediction for that patient.

3. **A near-zero *signed* mean SHAP means two different things.** For a linear
   model it is ~0 *by construction* (the baseline is the centred mean input), so
   the direction is read from the fitted coefficient sign instead. For a tree
   model it means the effect genuinely differs between patients, and the code
   labels it "effect varies by patient" rather than inventing a direction.

Additivity is asserted, not assumed: `base + sum(contributions)` must reproduce
the model's own output to within 1e-4, checked for every explained case and
across the whole explanation sample.

In [ ]:
exp = json.loads((prep.MODELS_DIR / "explainability" / "explainability_summary.json")
                  .read_text(encoding="utf-8"))
additivity = pd.DataFrame([
    {"model": PRETTY[m], "verified against": r["output_compared_against"],
     "scale": r["declared_output_scale"], "max gap": r["max_abs_gap"]}
    for m, r in exp["additivity_check"].items()
])
additivity

In [ ]:
coefficients = pd.read_csv(prep.MODELS_DIR / "explainability" / "logistic_regression_coefficients.csv")
coefficients.head(12)[["feature", "coefficient", "direction"]].round(4)

In [ ]:
shap_scales = {m: exp["output_scale"][m] for m in exp["output_scale"] if m != "note"}
for model_name in ["random_forest", "xgboost"]:
    print(f"--- {PRETTY[model_name]} global SHAP ---")
    print(f"    {shap_scales[model_name]}")
    table = pd.read_csv(prep.MODELS_DIR / "explainability" / f"{model_name}_shap_global.csv")
    print(table.head(10)[["feature", "mean_abs_shap", "direction"]].round(4).to_string(index=False))
    print()

![Global feature importance, mean |SHAP| per model. Panels are not on a shared scale.](../models/explainability/global_feature_importance.png)

*Global feature importance, mean |SHAP| per model. Panels are not on a shared scale.*

![SHAP beeswarm, Logistic Regression (log-odds units).](../models/explainability/shap_summary_logistic_regression.png)

*SHAP beeswarm, Logistic Regression (log-odds units).*

![SHAP beeswarm, XGBoost (probability units).](../models/explainability/shap_summary_xgboost.png)

*SHAP beeswarm, XGBoost (probability units).*

`missingindicator_ca` ranks first for Logistic Regression - *above* `ca`
itself. That is not an artefact: the model learned that the **absence** of an
angiographic measurement carries its own signal, because rows without `ca` have
higher disease prevalence (59.6% vs 46.8%). The missingness indicator earns its
place in the pipeline on evidence, not by convention.

**Reading this honestly.** `cp_asymptomatic`, `exang`, `oldpeak` and `ca`
appear near the top for all three models, which is the useful signal here: three
different model families agree on which inputs they lean on. The `sex_Female`
dummy ranks high in Logistic Regression and XGBoost because `sex_Male` is the
reference level - the two dummies are one piece of information split across two
columns, and their coefficients must be read together.

Feature attribution describes **how a fitted model used its inputs**. It is not
evidence of medical causation, not a diagnostic finding, and correlated features
can share or dilute attribution between themselves. No individual explanation
below is a clinical finding about that person.

### Individual explanations

Cases are selected using **out-of-fold** scores, so a false positive is a real
error from a model that never saw that row - not a row a fully-fitted pipeline
has already memorised. Five per model: one TP, one TN, one FP, one FN, and one
high-missingness case.

In [ ]:
records = []
for entry in exp["individual_explanations"]:
    top = entry["top_contributors"][0]
    records.append({
        "model": PRETTY[entry["model"]],
        "case": entry["case"],
        "observed": "disease" if entry["observed_target"] == 1 else "no disease",
        "base": round(entry["base_value_expected_output"], 3),
        "sum of SHAP": round(entry["sum_of_contributions"], 3),
        "model output": round(entry["model_output"], 3),
        "top contributor": top["feature"],
        "its SHAP": round(top["contribution"], 3),
    })
individuals = pd.DataFrame(records)
print(f"scale: log-odds for Logistic Regression and XGBoost, probability for Random Forest")
print("base + sum(SHAP) reproduces model output in every row:",
      bool(np.allclose(individuals["base"] + individuals["sum of SHAP"],
                       individuals["model output"], atol=1e-3)))
individuals

Two individual explanations per model, kept as full waterfalls in
`models/explainability/`:

![Logistic Regression - a false positive, in log-odds. The features that pushed the score toward disease despite the observed outcome.](../models/explainability/shap_individual_logistic_regression_FP.png)

*Logistic Regression - a false positive, in log-odds. The features that pushed the score toward disease despite the observed outcome.*

![XGBoost - a true positive, in log-odds.](../models/explainability/shap_individual_xgboost_TP.png)

*XGBoost - a true positive, in log-odds.*

The interesting part of a false positive is not the top driver but the
features that pushed the *same* way while the observed outcome went the other
way. In the Logistic Regression false positive, `exang`, `thal_reversable
defect` and `cp_asymptomatic` all contributed toward disease and the patient had
no disease. Attribution explains the score; the score was wrong here, and the
explanation is faithful to that wrongness. That is the honest reading - a
flattering story about a patient would be a misreading of the same plot.

`base + sum(contributions)` reproducing the model's own output in the table
above is the arithmetic check that the plot is describing the real prediction
and not a decorative approximation of it.

## 13. The locked holdout, opened once

Everything above used the 736 training rows only. These 184 rows were not
scored, plotted, tuned or thresholded until every decision was frozen, and they
were not consulted when choosing a threshold, a feature set or a model.

They are reported for completeness and to check the development estimates were
not optimistic.

In [ ]:
holdout = pd.read_csv(prep.MODELS_DIR / "holdout_final_metrics.csv")
holdout[["model", "n", "n_positive", "prevalence", "threshold", "roc_auc", "pr_auc",
         "recall", "precision", "f1", "accuracy", "tn", "fp", "fn", "tp"]].round(4)

In [ ]:
comparison = tuned_overall[["model", "roc_auc_mean", "pr_auc_mean", "f1_mean"]].merge(
    holdout[["model", "roc_auc", "pr_auc", "f1"]], on="model"
)
comparison["roc_auc gap (holdout - cv)"] = (comparison["roc_auc"] - comparison["roc_auc_mean"]).round(4)
comparison["pr_auc gap (holdout - cv)"] = (comparison["pr_auc"] - comparison["pr_auc_mean"]).round(4)
comparison.round(4)

**Reading this honestly.** Holdout ROC-AUC lands within 0.005 of the
cross-validation mean for every model, in both directions across metrics. That
is reassuring about the *procedure*, not about the models: it says the
cross-validation numbers were not inflated, on a 184-row split that is itself
too small to resolve differences between three models.

The holdout does not rescue any conclusion. It cannot be re-opened.

## 14. Verification suite

assertions covering reproducibility, leakage safety, preprocessing
correctness, artefact completeness, metric hygiene, and dependency pinning.
The exact count is printed below rather than hard-coded here.
Run standalone with `python -m src.verify_stage3`.

In [ ]:
report = verify_stage3.run_all(verbose=True)
print()
print("all checks passed:", report["all_passed"],
      f"({report['passed']}/{report['total']})")

## 15. Limitations

1. **920 rows, four cohorts.** Every metric carries wide uncertainty. After the
   paired test, two of three model pairs are no longer separated on ROC-AUC at
   all, so this notebook does not rank the three models as clinically better or
   worse; it reports the gap and its paired spread.
2. **Cohorts are not clinically comparable.** Prevalence spans 36.2% to 93.9%.
   Pooled metrics average over populations that differ in referral pathway and
   measurement, and per-cohort figures on the small slices are flagged.
3. **`ca` is an angiographic measurement** available for one cohort. The
   ablation measures its contribution; it does not resolve how to obtain it.
4. **One search budget, one repeat for search.** Random Forest and XGBoost
   searches covered 20 of 288 and 20 of 1296 combinations. A different draw
   could select different parameters.
5. **No external validation.** These are in-sample-of-this-dataset research
   scores. Transport to a new population, a different measurement protocol or a
   different referral pathway is unestablished.
6. **Threshold and calibration are reported, not chosen.** Both need a clinical
   and economic context this dataset does not contain.
7. **Feature attribution is model behaviour, not medical knowledge.** Correlated
   features share and dilute attribution; no explanation here is a clinical
   finding.
8. **Research use only.** Not a medical device, not a diagnostic system, not
   validated for clinical use, and requires external validation before any
   deployment.

## 16. Required wording

These are the phrases this project's results must be described with. They are
enforced in the report cards rather than left to prose:

* a **research prototype** for health **risk assessment**
* model **discrimination** rather than accuracy claims
* outputs are **research risk scores**, not clinically validated probabilities
* **requires external validation** before any clinical use
* no causal language for feature attribution or error analysis

## 17. Artefacts written by this stage

| File | Contents |
|------|----------|
| `cv_metrics_overall.csv` | repeated CV, un-tuned baselines, mean +/- SD |
| `cv_metrics_per_fold.csv` / `cv_metrics_tuned_per_fold.csv` | per-fold rows, so model-vs-model comparisons can be paired |
| `cv_metrics_tuned_overall.csv` | repeated CV, selected configurations |
| `cv_metrics_by_cohort.csv` / `cv_metrics_tuned_by_cohort.csv` | per-cohort CV with reliability flags |
| `threshold_analysis.csv` | full 5-point grid, per model |
| `calibration_metrics.csv` / `calibration_bins.csv` / `calibration_curve.png` | calibration numbers and reliability diagram |
| `feature_ablation_ca.csv` / `feature_ablation_ca_by_cohort.csv` | `ca` availability ablation |
| `holdout_final_metrics.csv` | the single locked holdout evaluation |
| `tuned/*.joblib` | selected-configuration pipelines, preprocessing included |
| `explainability/` | coefficients, importances, SHAP tables, plots, individual cases |
| `stage3_report.json` | consolidated machine-readable record |

Nothing here is deployed. No API, no dashboard, no service.

In [ ]:
print("raw CSV unchanged:", prep.file_sha256() == prep.EXPECTED_SHA256)
print("expected:", prep.EXPECTED_SHA256)
print()
print("artefacts in models/:")
for p in sorted(prep.MODELS_DIR.rglob("*")):
    if p.is_file() and p.suffix in {".csv", ".json", ".joblib", ".png"}:
        print(f"  {p.relative_to(PROJECT_ROOT)}")